# Infaction World Generator v0.8.0 — Locked Zone Framework v0.4.0
Unique Zone Type, strict physical Inside majority, contextual Secret placement, simple parent compatibility. Shared JavaScript source; historical notebooks retained.


In [ ]:
import urllib.request, pathlib, subprocess, json, shutil
if not shutil.which('node'):
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','nodejs'],check=True)
url = 'https://raw.githubusercontent.com/playrenathea/rantau/main/world-generator.js'
pathlib.Path('world-generator.js').write_bytes(urllib.request.urlopen(url).read())
def generate(seed=42):
assert json.loads(subprocess.check_output(['node','-e','console.log(JSON.stringify(require(\"./world-generator.js\").version))'],text=True)) == '0.8.0', 'Generator version mismatch'
    return json.loads(subprocess.check_output(['node','-e',f'console.log(JSON.stringify(require("./world-generator.js").generate({int(seed)})))'],text=True))
print(json.dumps(generate(),indent=2))


In [ ]:
# Inspect a world in a readable form
world = generate(42)
print(world['city'])
factions = {f['id']: f['name'] + ' (' + ' + '.join(f['types']) + ')' for f in world['factions']}
for region in ['Inside City','Outside City','Secret Zone']:
    print('\n' + region)
    for z in world['zones']:
        if z['region'] == region:
            print(z['id'], z['label'], '|', factions.get(z['factionId']), '|', ', '.join(z['functions']), '|', z['reason'], '| physical:', z['physicalRegion'], '| host:', z['parentId'])


In [ ]:
# Batch validation in one Node process
N = 1000
script = '''const G=require('./world-generator.js');let reg={},types={},bad=0,total=0;for(let i=0;i<%d;i++){const w=G.generate(i),z=w.zones;for(const q of z){reg[q.region]=(reg[q.region]||0)+1;types[q.type]=(types[q.type]||0)+1;total++}if(z.length<5||z.length>8||new Set(z.map(q=>q.type)).size!==z.length||z.filter(q=>q.physicalRegion==='Inside City').length<=z.length/2||!z.some(q=>q.region==='Outside City')||['Authority','Commerce','Residence'].some(k=>w.zoneCoverage['Inside City'][k]<1))bad++}console.log(JSON.stringify({universes:%d,total,reg,types,bad}))''' % (N,N)
result = json.loads(subprocess.check_output(['node','-e',script],text=True))
assert result['bad'] == 0
print(json.dumps(result,indent=2))
